In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from grid_cells.random_walk import generate_bat_flight
from grid_cells.attractor_networks import HeadDirection
from grid_cells.plot_tools import angular_error, get_plot_grid, activity_map

import tqdm
import os

DATA_DIR = "../simulation_data"
PLOTS_DIR = "../plots"

In [ ]:
T = 400
dt = 0.5e-3
n = 64
moon_position = np.array([0, 1, 1]) / np.sqrt(2)

In [ ]:
net = HeadDirection(n=n, dt=dt, use_single_bump=True, intrinsic_noise=0)


def make_anchor_weight(anchor, sigma=0.25):
    def weight_func(position):
        err = angular_error(position, anchor)
        d2 = np.sum(err**2)
        return np.exp(-d2 / (2 * sigma**2))

    return weight_func


anchors = np.array([[0, 0]])
for anchor in anchors:
    net.add_anchor_point(
        weight_func=make_anchor_weight(anchor, sigma=0.25),
        strength=0.2,
        mask=net.encode_orientation(anchor),
    )


net.s[0, 0] = 10
net.warm_up(pos=np.array([0, 0]))

In [ ]:
## Simulation parameters:
n_min_plots = 10
dt = 0.5e-3
n = 128

net = HeadDirection(n=n, dt=dt, use_single_bump=True)

recordings_with_anchor = []
n_anchors = 0


def make_anchor_weight(anchor, sigma=0.25):
    def weight_func(position):
        err = angular_error(position, anchor)  # shape (2,), wraparound-safe
        d2 = np.sum(err**2)
        return np.exp(-d2 / (2 * sigma**2))

    return weight_func


anchors = np.array([np.pi, np.pi])[np.newaxis, :]
for anchor in anchors:
    net.add_anchor_point(
        weight_func=make_anchor_weight(anchor, sigma=0.2),
        strength=0.1,
        mask=net.encode_orientation(anchor, width=0.2),
    )


error = 0.5
net.s = net.encode_orientation([np.pi - error, np.pi])
net.warm_up()
ncols = np.int64(np.ceil(np.sqrt(n_min_plots)))
nrows = np.int64(np.ceil(n_min_plots / ncols))
n_plots = nrows * ncols
fig, ax = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
ax = ax.flatten()
n_steps = 250
records = np.linspace(0, n_steps - 1, n_plots, dtype=int)
plot_counter = 0
time = 0


for step_iter in range(n_steps):
    if step_iter == records[plot_counter]:
        ax[plot_counter].imshow(net.s, origin="lower")
        phi_decoded = net.decode_orientation()[0]
        ax[plot_counter].set_title(r" $\phi_{dec}" + f"={phi_decoded/np.pi:.2f}$")
        plot_counter += 1
    net.step(0, 0, pos=np.array([np.pi - error / 3, np.pi - error / 3]))
    time += dt